# Human in Loop

In [ ]:
import os
import requests
from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv(usecwd=True))

openweather_key = os.getenv("OPENWEATHER_API_KEY")
geo_url = "https://api.openweathermap.org/geo/1.0/direct"
weather_url = "https://api.openweathermap.org/data/2.5/weather"
resend_api_key = os.getenv("RESEND_API_KEY")

llm = ChatOpenAI(
    model=os.getenv("model", "openai/gpt-oss-120b"),
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    max_tokens=700,
)
print("Model:", os.getenv("model", "openai/gpt-oss-120b"))
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING", "off"))

request="check the weather in Tokyo and email me a short weather report"

Two helper functions to print same way 

    show-prints each toolcall
    pending-print what the paused agent is waiting for: tool,arguments,your decision

In [ ]:
def show(result):
    for message in result["messages"]:
        if message.type=="ai" and message.tool_calls:
            for call in message.tool_calls:
                print(f"MODEL->{call['name']}({call['args']})")
        elif message.type=="tool":
            print(f"TOOL <- [{message.status}] {message.content[:150]}")
    if "__interrupt__" in result:
        print("PAUSED: waiting for human decision")
    else:
        print(f"ANSWER: {result['messages'][-1].content}")

# def pending(result):
#     interrupt = result["__interrupt__"][0]
#     request=interrupt.value
#     action=request["action_requests"][0]
#     print("WAITING TO HUMAN APPROVAL:",action["name"])
#     for name,args in action["args"].items():
#         print(f"{name}: {args}")
#     print("you may:",request["review_configs"][0]["allowed_decisions"])
#     return action

def pending(result):
    if "__interrupt__" not in result:
        print("No pending human approval.")
        for message in result.get("messages", []):
            print("\nMESSAGE TYPE:", message.type)
            print("CONTENT:", message.content)
            tool_calls = getattr(message, "tool_calls", [])
            if tool_calls:
                for call in tool_calls:
                    print("TOOL:", call["name"])
                    print("ARGS:", call["args"])
        return None
    interrupt = result["__interrupt__"][0]
    request_data = interrupt.value
    action = request_data["action_requests"][0]
    print("\nWAITING FOR HUMAN APPROVAL:", action["name"])
    for key, value in action["args"].items():
        print(f"{key}: {value}")
    print("\nAVAILABLE DECISIONS:",request_data["review_configs"][0]["allowed_decisions"])
    return action

# Two tools

1.get_weather- get data about weather

2.send_email- sends a real email through resend

In [ ]:
to_email="neelaganeshd@gmail.com"
from_email=os.getenv("RESEND_FROM_EMAIL","Atlas <onboarding@resend.dev>")

@tool
def get_weather(city:str)->str:
    """Get the weather in a city"""
    place=requests.get(f"{geo_url}?q={city}&limit=1&appid={openweather_key}",timeout=10).json()[0]
    params={"lat":place["lat"],"lon":place["lon"],"appid":openweather_key,"units":"metric"}
    data=requests.get(weather_url,params=params,timeout=10).json()
    return f"{place["name"]}: {data["weather"][0]["description"], {data["main"]["temp"]}}°C"

@tool
def send_email(subject:str,body:str)->str:
    """Send an email through resend"""
    response=requests.post(f"https://api.resend.com/emails",timeout=15,headers={"Authorization":f"Bearer {resend_api_key}"},json={"from":from_email,"to":to_email,"subject":subject,"html":body})
    return f"Email sent to {to_email} (id: {response.json()['id']})"

# no human in loop

In [ ]:
plain_agent=create_agent(llm,tools=[get_weather, send_email])
request="check the weather in vizag and email me a short weather report"

result=plain_agent.invoke({"messages":[{"role":"user","content":request}]},config={"run_name":"4.17 no human"})
show(result)

# adding middleware

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

hitl_agent=create_agent(
    llm,
    tools=[get_weather,send_email],
    system_prompt="you are a helpful assistent,do complete the entire tasks using info",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True,"get_weather": False})],
    checkpointer=InMemorySaver(),
)
request="check the weather in london and email me a short weather report"
config={"configurable":{"thread_id":"approve_demo"},"run_name":"4.17 approve"}
result=hitl_agent.invoke({"messages":[{"role":"user","content":request}]},config)
show(result)
action=pending(result)

In [ ]:
result

# 1.Approve

In [ ]:
result=hitl_agent.invoke(Command(resume={"decisions":[{"type":"approve"}]}),config)
show(result)

# 2.Edit


In [ ]:
config={"configurable":{"thread_id":"edit_demo"},"run_name":"4.17 edit"}
request="check the weather in newwork and email me a short weather report"
result=hitl_agent.invoke({"messages":[{"role":"user","content":request}]},config)
show(result)
action=pending(result)

In [ ]:
new_args={
    "subject":"newwork weather (checked by human)",
    "body":action["args"]["body"]+"\n\n Reviewed by ganesh before sending"}
decision={"type":"edit","edited_action":{"name":action["name"],"args":new_args}}
result=hitl_agent.invoke(Command(resume={"decisions": [decision]}),config)
show(result)

# 3.Reject


In [ ]:

config={"configurable":{"thread_id":"reject_demo"},"run_name":"4.17 reject"}
request="check the weather in newwork and email me a short weather report"
result=hitl_agent.invoke({"messages":[{"role":"user","content":request}]},config)
show(result)
action=pending(result)

In [ ]:
decision={"type":"reject","message":"don't sent this email,i'll share myself"}
result=hitl_agent.invoke(Command(resume={"decisions": [decision]}),config)
show(result)

# 4.choose which decisions are allowed

In [ ]:

strict_agent=create_agent(
    llm,
    tools=[get_weather,send_email],
    system_prompt="you are a helpful assistent,do complete the entire tasks using info",
    middleware=[HumanInTheLoopMiddleware(interrupt_on={
        "send_email": {"allowed_decisions":["approve","reject"]},
        "get_weather": False})],
    checkpointer=InMemorySaver(),
)
request="check the weather in hongkong and email me a short weather report"
config={"configurable":{"thread_id":"strict_demo"},"run_name":"4.17 no edit"}
result=strict_agent.invoke({"messages":[{"role":"user","content":request}]},config)
show(result)
action=pending(result)

# 4.1 Approve

In [ ]:
strict_agent.invoke(Command(resume={"decisions": [{"type":"approve"}]}),config)

# 5.check first and then resume 

In [ ]:
def resume_if_allowed(agent,result,dict,config):
    allowed=result["__interrupt__"][0].value["review_configs"][0]["allowed_decisions"]
    if decision["type"] not in allowed:
        print(f"not sent to agent: {decision["type"]} is not available,you may: {allowed}")
        return result
    return agent.invoke(Command(resume={"decision":[decision]}),config)


config={"configurable":{"thread_id":"strict_demo-checked"},"run_name":"4.17 check before resume"}
result=strict_agent.invoke({"messages":[{"role":"user","content":request}]},config)
action=pending(result)

edit={"type":"edit","edited_action":{"name":"send_email","args":{"subject":"hi","body":"edited"}}}
result=resume_if_allowed(strict_agent,result,edit,config)
result=resume_if_allowed(strict_agent,result,{"type":"approve"},config)
show(result)

# two real reviewers [easy way]
reviewer 1: type your answer

In [ ]:
def review_with_input(agent,config):
    result=agent.invoke({"messages":[{"role":"user","content":request}]},config)
    while "__interrupt__" in result:
        action=pending(result)
        choice=input("a = approve, e = edit, r = reject").strip().lower()
        if choice=="e":
            args={"subject":input("new subject:"),"body":input("new body:")}
            decision={"type":"edit","edited_action":{"name":action["name"],"args":args}}
        elif choice=="a":
            decision={"type":"approve"}
        else:
            reason=input("why not?") or "the reviewer rejected this mail"
            decision={"type":"reject","message":reason}
        result=agent.invoke(Command(resume={"decisions":[decision]}),config)
    show(result)

review_with_input(hitl_agent,{"configurable":{"thread_id":"input_demo"},"run_name":"4.17 input review"})

 # two real reviewers [easy way] 
review 2: click on button


In [ ]:
import ipywidgets as widgets
from IPython.display import display

button_config = {"configurable": {"thread_id": "buttons-demo"}, "run_name": "4.17 buttons reviewer"}
result = hitl_agent.invoke({"messages": [{"role": "user", "content": request}]}, button_config)
action=pending(result)

subject_box=widgets.Text(value=action["args"]["subject"],description="Subject",layout={"width":"600px"})
body_box=widgets.Textarea(value=action["args"]["body"],description="Body",layout={"width":"600px"})
reason_box=widgets.Text(placeholder="Why not?",description="Reason",layout={"width":"600px"})
approve_button=widgets.Button(description="Approve",button_style="success")
edit_button=widgets.Button(description="Edit & send",button_style="info")
reject_button=widgets.Button(description="Reject",button_style="danger")
output=widgets.Output()

In [ ]:
def decide(decision: dict) -> None:
    for button in (approve_button, edit_button, reject_button):
        button.disabled= True
    with output:
        show(hitl_agent.invoke(Command(resume={"decisions": [decision]}), button_config))
    
approve_button.on_click(lambda _: decide({"type": "approve"}))
edit_button.on_click(lambda _: decide({"type": "edit", "edited_action": {
    "name": "send_email"},"args": {"subject": subject_box.value, "body": body_box.value}}))
reject_button.on_click(lambda _: decide({"type": "reject","message": reason_box.value or "The reviewer rejected this email."}))

display(subject_box, body_box, reason_box, widgets. HBox([approve_button, edit_button, reject_button]), output)